In [2]:
import os
import shutil

import kagglehub
import pandas as pd

if not os.path.isdir("nlp-getting-started"):
    path = kagglehub.competition_download("nlp-getting-started")
    shutil.move(path, "nlp-getting-started")

df_train = pd.read_csv("nlp-getting-started/train.csv")
df_test = pd.read_csv("nlp-getting-started/test.csv")

df_train

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


In [21]:
import re
import gzip
import json
import time
import uuid

from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from geopy.geocoders import Nominatim
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def tokenize(doc):
     return re.findall(r"\w+", doc) # TODO: make better tokenizer

class D2VVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, **kwargs):
        self.model = None
        self.hyperparameters = kwargs
    
    def fit(self, X, y=None):
        print(X, type(X))
        self.model = Doc2Vec([TaggedDocument(tokenize(x), [i]) for i, x in enumerate(X)], **self.hyperparameters)
        return self

    def transform(self, X, y=None):
        print(X, type(X))
        return np.vstack([self.model.infer_vector(tokenize(x)) for x in X])

class GeopyLocator(BaseEstimator, TransformerMixin):
    def __init__(self, **kwargs):
        try:
            with gzip.open("geopy_cache.json.gz", "rt") as file:
                self.cache = json.load(file)
        except FileNotFoundError:
            self.cache = {}
            
        self.geolocator = Nominatim(user_agent="ece_6420_final", timeout=10)
    
    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        locations = []
        dirty = False
        
        try:
            for x in X:
                if x != x: # NaN
                    locations.append([np.nan, np.nan])
                elif x in self.cache:
                    if self.cache[x] is None:
                        locations.append([np.nan, np.nan])
                    else:
                        locations.append(self.cache[x])
                else:
                    print("Geolocating \"{}\"...".format(x))
                    if x.strip(): # Special case: geolocator crashes on all-whitespace strings
                        location = self.geolocator.geocode(x)
                    else:
                        location = None
                    dirty = True
                    time.sleep(1.5) # Respect Nominatim usage policy

                    if location is None:
                        self.cache[x] = None
                        locations.append([np.nan, np.nan])
                    else:
                        self.cache[x] = [location.latitude, location.longitude]
                        locations.append([location.latitude, location.longitude])
                        
            return np.array(locations)
        finally:
            if dirty:
                with gzip.open("geopy_cache.json.gz", "wt") as file:
                    json.dump(self.cache, file)
    
ct = make_column_transformer(
    ("drop", ["id"]),
    (OneHotEncoder(drop=[np.nan]), ["keyword"]),
    (make_pipeline(GeopyLocator(), SimpleImputer(), StandardScaler()), "location"),
    (D2VVectorizer(), "text")
)

ct.fit_transform(df_train)

Geolocating "  "...
Geolocating "Kentucky, USA"...
Geolocating "New Jersey/New York"...
Geolocating "#SandraBland"...
Geolocating "'Merica"...
Geolocating "Harper Woods, MI"...
Geolocating "dubai "...
Geolocating "staggering on tenement roofs"...
Geolocating "houston"...
Geolocating "Orlando "...
Geolocating "Noida, NCR, India"...
Geolocating "MÌ©xico"...
Geolocating "NYC / International"...
Geolocating "PA, USA"...
Geolocating "call me peach or sam lo"...
Geolocating "LITTLETON, CO, USA, TERRAN"...
Geolocating "Orlando"...
Geolocating "shoujo hell "...
Geolocating "10 Steps Ahead.  Cloud 9"...
Geolocating "central chazifornia"...
Geolocating "????s ?? ????Ìø????Ì¡a"...
Geolocating "T E X A S | wwat 8.24.14"...
Geolocating "876 Jamrock."...
Geolocating "å_å_Los Mina Cityã¢"...
Geolocating "St Louis, MO"...
Geolocating "WORLDWIDE!"...
Geolocating "Greensburg, PA"...
Geolocating "Like us on Face "...
Geolocating "Montgomery County, MD"...
Geolocating "Cambridge, Massachusetts, U.S."...


Geolocating "calgary,ab"...
Geolocating "Newton, NJ 07860"...
Geolocating "she/her/your majesty/empress"...
Geolocating "LYNBROOK"...
Geolocating "Argus Industries \m/666\m/"...
Geolocating "Friendswood, TX"...
Geolocating "Sugar Land, TX"...
Geolocating "(a) property of the universe"...
Geolocating "Home is where we park it!"...
Geolocating "Webster, TX"...
Geolocating "Maracay y Nirgua, Venezuela"...
Geolocating "North Cack/919"...
Geolocating "Scottsdale, AZ"...
Geolocating "Victoria, Canada"...
Geolocating "FILM OUT LATE 2015"...
Geolocating "Miami,Fla"...
Geolocating "santo domingo"...
Geolocating "Worldwide - Global"...
Geolocating "Suva, Fiji Islands."...
Geolocating "Americas Newsroom"...
Geolocating "Yogya Berhati Nyaman"...
Geolocating "Fredericksburg, Virginia"...
Geolocating "Wherever I'm needed"...
Geolocating "Charlotte "...
Geolocating "Sale, England"...
Geolocating "Newcastle, England "...
Geolocating "Kashmir!"...
Geolocating "in my head"...
Geolocating "Rutherfordton,

array([[ 0.        ,  0.        ,  0.        , ..., -0.01655749,
         0.01285915, -0.00126081],
       [ 0.        ,  0.        ,  0.        , ..., -0.03858704,
         0.00157049,  0.02024019],
       [ 0.        ,  0.        ,  0.        , ..., -0.04740846,
         0.02223838,  0.00671396],
       ...,
       [ 0.        ,  0.        ,  0.        , ..., -0.08725831,
        -0.01011566,  0.04537441],
       [ 0.        ,  0.        ,  0.        , ..., -0.04309445,
         0.01445794,  0.01697868],
       [ 0.        ,  0.        ,  0.        , ..., -0.05502802,
        -0.00079516,  0.02821686]], shape=(7613, 323))

In [22]:
ct.fit_transform(df_test)

Geolocating "Niall's place | SAF 12 SQUAD |"...
Geolocating "Los Angeles, Califnordia"...
Geolocating "threeonefive. "...
Geolocating "Whoop Ass, Georgia"...
Geolocating "scarborough, ontario"...
Geolocating "121 N La Salle St, Suite 500"...
Geolocating "Wandering"...
Geolocating "Homewood, PA"...
Geolocating "Bexhill"...
Geolocating "Anime World"...
Geolocating "Cowtown, Caliii !!"...
Geolocating "All Motorways, UK"...
Geolocating "Frankfurt, Germany"...
Geolocating "Gresham, OR"...
Geolocating "Las Vegas "...
Geolocating "Midland, Mi"...
Geolocating "SÌ£o Vicente, SÌ£o Paulo"...
Geolocating "Vault 101, Fallout"...
Geolocating "Davis, CA"...
Geolocating "In your hearts and minds"...
Geolocating "Runcorn"...
Geolocating "Lake Charles, LA"...
Geolocating "inland empire ca"...
Geolocating "Muscat"...
Geolocating "USA - Global Online Sales "...
Geolocating "Istanbul, TÌ_rkiye"...
Geolocating "High in Prague with Aya"...
Geolocating "born on september 1st"...
Geolocating "New Orleans"...
G

Geolocating "SW London (RBK)"...
Geolocating "San Diegohhjhhhhhghghpjg"...
Geolocating "SQU\/D, uk"...
Geolocating "$$$"...
Geolocating ";)"...
Geolocating "Ellixton London's 33rd Borough"...
Geolocating "Ngayogyakarta Hadiningrat "...
Geolocating "San Francisco, Los Angeles"...
Geolocating "Arizona, USA"...
Geolocating "Canada | #LUX"...
Geolocating "among the socially awkward ?"...
Geolocating "West Hampstead, London NW6"...
Geolocating "Paranaque City, National Capital Region"...
Geolocating "Toronto  #6ixSideMafia"...
Geolocating "jonas/lovato/bieber/5sos"...
Geolocating "Carrboro, NC"...
Geolocating "#BVSTRONG"...
Geolocating "Blackfield, England"...
Geolocating "united states"...
Geolocating "Cape Coral Fl"...
Geolocating "Leanbox?"...
Geolocating "Mechanicsville, VA"...
Geolocating "Edmonton"...
Geolocating "www.facebook.com/Randirobics"...
Geolocating "OHIO"...
Geolocating "? In your head ?"...
Geolocating "hyejeong?taehyung"...
Geolocating "Emirate"...
Geolocating "Twin Cities

Geolocating "real world"...
Geolocating "130515 Û¢ Gallavich."...
Geolocating "Bangalore,India"...
Geolocating "Pico Rivera, CA"...
Geolocating "Playing: HL2: EP1, Dust: AET"...
Geolocating "Mtl"...
Geolocating "East la Mirada, CA"...
Geolocating "Louisville Kentucky"...
Geolocating "Jeddah, Makkah Al Mukarrama"...
Geolocating "Do Not Follow Me, Am I a Bot."...
Geolocating "Bay Area, California "...
Geolocating "Rocky Mountains Colorado"...
Geolocating "San Bernardo - Chile"...
Geolocating "Duel Academia"...
Geolocating "   Manchester/Nantwich"...
Geolocating "Under the rain..."...
Geolocating "clearlake"...
Geolocating "Twitter."...
Geolocating "beyond time and space"...
Geolocating "jds "...
Geolocating "Milan, Lombardy"...
Geolocating "Metro, Lampung ~ Balikpapan"...
Geolocating "Bieber Fever UK"...
Geolocating "32935"...
Geolocating "301|804"...
Geolocating "Gap"...
Geolocating "Stone Ridge, VA"...
Geolocating "Staffordshire"...
Geolocating "Central #Ohio"...
Geolocating "hayling 

Geolocating "| INDIA |"...
Geolocating "Pune"...
Geolocating "chapel hill, cary n.c."...
Geolocating "Glendale, AZ"...
Geolocating "Covering the U.S.A."...
Geolocating "Fantasy Land"...
Geolocating "Kibworth Beauchamp, England"...
Geolocating "3/12 + Jake Miller"...
Geolocating "www.dorsavi.com"...
Geolocating "All Round The World"...
Geolocating "Bellingham, WA"...
Geolocating "Searching for Rivendell"...
Geolocating "Shrewsbury, England"...
Geolocating "GroÌÙdeutsches Reich"...
Geolocating "Beijing,China"...
Geolocating "RELEASE THE REIS  "...
Geolocating "That London"...
Geolocating "#RoshanPakistan"...
Geolocating "Wickford"...
Geolocating "Central New Jersey"...
Geolocating "Shangri-La"...
Geolocating "Morgantown, WV"...
Geolocating "Upper St. Clair"...
Geolocating "trinity nc"...
Geolocating "Budgam kashmir"...
Geolocating "somewhere over the rainbow"...
Geolocating "Northwest Arkansas"...
Geolocating "Staten Island, NY"...
Geolocating "NWA & River Valley"...
Geolocating "Winston

Geolocating "probably bargos"...
Geolocating "Moreno City - Buenos Aires"...
Geolocating "Salem, Oregon"...
Geolocating "Franklinton - BR - Houston"...
Geolocating "Roseland"...
Geolocating "Barnsley"...
Geolocating "IL?MI"...
Geolocating "your mom"...
Geolocating " "...
Geolocating "bliss"...
Geolocating "@rejxctedmgc is my Harry?"...
Geolocating "kentucky"...
Geolocating "ig: j.nessaa"...
Geolocating "madrid"...
Geolocating "NJ, USA"...
Geolocating "The Land of Make Believe"...
Geolocating "Alberta, Canada"...
Geolocating "Saint John, N.B, Canada "...
Geolocating "Northeast United States "...
Geolocating "New England ????"...
Geolocating "AUSTRALIA"...
Geolocating "Florence, SC"...
Geolocating "Sunmy Melbourne, England"...
Geolocating "424 N. FAIRFAX AVE. 90036"...
Geolocating "Fairbanks Alaska"...
Geolocating "Where is my mind?"...
Geolocating "North Coast of O-H-I-O"...
Geolocating "Oman "...
Geolocating "BLF (MP)/ PTA (GP)/ PHB(LIMPS)"...
Geolocating "Bedford Stuyvesant, Brooklyn"

array([[ 0.        ,  0.        ,  0.        , ..., -0.0147693 ,
         0.01113956, -0.0115328 ],
       [ 0.        ,  0.        ,  0.        , ..., -0.02537214,
         0.0154197 , -0.023324  ],
       [ 0.        ,  0.        ,  0.        , ..., -0.02519361,
         0.02118589, -0.01837625],
       ...,
       [ 0.        ,  0.        ,  0.        , ..., -0.01337525,
         0.00220417, -0.01034906],
       [ 0.        ,  0.        ,  0.        , ..., -0.01395831,
         0.00804868, -0.0035955 ],
       [ 0.        ,  0.        ,  0.        , ..., -0.00617199,
         0.00758176, -0.00198097]], shape=(3263, 323))